# Create Multiscale Zarr Pyramid for Web Visualization

This notebook builds a multiscale Zarr pyramid from the MODIS snow phenology Icechunk store so the dataset can be visualized as a slippy web map using [zarr-layer](https://github.com/carbonplan/zarr-layer) without creating a separate visualization copy or running a tile server.

**Stack**
- [topozarr](https://github.com/carbonplan/topozarr): generates coarsened multi-resolution levels following the [GeoZarr multiscales spec](https://github.com/zarr-developers/geozarr-spec)
- [zarr-layer](https://github.com/carbonplan/zarr-layer): TypeScript library that fetches and renders Zarr as a custom MapLibre/Mapbox layer, reprojecting on the GPU on the fly

**Why multiscales?**  
The full store is 86 400 × 43 200 pixels. Without overview levels, zarr-layer would need to fetch the entire array at every zoom level. Multiscales let the viewer fetch only the appropriate coarsened level for the current viewport — qualitatively matching the performance of a traditional tile server.

**Projection note**  
The store is in MODIS Sinusoidal, *not* Web Mercator. zarr-layer reprojects on the GPU client-side, so we can write the pyramid in native projection and preserve pixel fidelity.

---
**Dependencies**: `topozarr` and `xproj` are added to `pixi.toml` as PyPI dependencies. Run `pixi install` before launching this notebook.

## Imports

In [1]:
from pathlib import Path
import time
import sys
import pandas as pd
import xarray as xr
import numpy as np
import zarr
import icechunk
import rioxarray
import xproj
from topozarr import create_pyramid, ZarrLayerVarConfig
import dask
from dask.diagnostics import ProgressBar
import adlfs
from obstore.store import AzureStore
from zarr.storage import ObjectStore
from IPython.display import JSON
import json

sys.path.insert(0, str(Path('..').resolve()))
from modis_snow_phenology import Config

config = Config('config/config_with_secrets_v1.txt')

# https://github.com/carbonplan/topozarr/blob/main/scripts/build_demo_data.py
# from https://github.com/carbonplan/ocr/blob/main/ocr/pipeline/create_pyramid.py
# zarr.config.set({'async.concurrency': 128})
#dask.config.set(scheduler='threads')
#dask.config.set(scheduler='threads', num_workers=32)
# zarr.config.set({'async.concurrency': 128})

from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=8,
    threads_per_worker=4,
    memory_limit='15GB',       # changed from 10 to 15GB
    local_directory='/tmp/dask-spill',
)
client = Client(cluster)

zarr.config.set({
    'async.concurrency': 128,   # 32 to 128GB
    'threading.max_workers': 16, # codec threads = CPU count
})

In [2]:
import os, psutil

print(f"CPUs: {os.cpu_count()}")
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.0f} GB")
print(f"Available RAM now: {psutil.virtual_memory().available / 1e9:.0f} GB")

ncores = client.nthreads()   # {worker_address: n_threads}
print(f"Workers:       {len(ncores)}")
print(f"Total threads: {sum(ncores.values())}")
print(f"Threads/worker: {set(ncores.values())}") 

print(f"Zarr config: async.concurrency: {zarr.config.get('async.concurrency')}")
print(f"Zarr config threading.max_workers: {zarr.config.get('threading.max_workers')}")

CPUs: 16
Total RAM: 134 GB
Available RAM now: 128 GB
Workers:       8
Total threads: 32
Threads/worker: {4}
Zarr config: async.concurrency: 128
Zarr config threading.max_workers: 16


In [3]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/egagli/proxy/8787/status,
Dashboard: /user/egagli/proxy/8787/status,Workers: 8
Total threads: 32,Total memory: 111.76 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41697,Workers: 0
Dashboard: /user/egagli/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:32783,Total threads: 4
Dashboard: /user/egagli/proxy/36945/status,Memory: 13.97 GiB
Nanny: tcp://127.0.0.1:45587,


## 1. Open the Icechunk Store

In [4]:
storage = icechunk.azure_storage(
    account=config.AZURE_STORAGE_ACCOUNT,
    container=config.AZURE_CONTAINER,
    prefix=config.ICECHUNK_PREFIX,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session('main')

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all',mask_and_scale=True)
ds

<xarray.Dataset> Size: 448GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
    spatial_ref           int64 8B ...
Data variables:
    SDD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SAD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 2. Prepare Dataset

topozarr needs:
1. A CRS assigned via `xproj` (`.proj.assign_crs()`)
2. The `spatial_ref` scalar coordinate removed — topozarr manages CRS metadata internally and the scalar causes issues during coarsening

Data comes out of the store as `float32` (Zarr's `mask_and_scale` replaces `int16` fill values with `NaN`). `create_pyramid(method='mean')` propagates NaNs correctly, so no extra masking is needed.

In [5]:
# Extract CRS WKT from the rioxarray spatial_ref before we drop it
crs = ds.rio.crs
crs

CRS.from_wkt('PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]')

In [6]:
# Drop the CF-convention spatial_ref scalar — xproj will carry the CRS instead
ds_clean = ds.drop_vars('spatial_ref')

# Assign CRS via xproj so topozarr can find it
ds_crs = ds_clean.proj.assign_crs(spatial_ref_crs={'wkt': crs.to_wkt()})
ds_crs

<xarray.Dataset> Size: 448GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
  * wkt                   int64 8B 0
Data variables:
    SDD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SAD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 149GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Indexes:
    wkt      CRSIndex (crs=PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom sphero ...)
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 3. Create Multiscale Pyramid

### Choosing the number of levels

Each level coarsens by 2× in both spatial dimensions. In topozarr's convention **level 0 is the finest (native resolution)** and level N_LEVELS−1 is the coarsest. The coarsest level should be small enough to fetch in a single request at the lowest zoom.

| N_LEVELS | Coarsest level | Coarsest shape (y × x) | Coarsest pixel size |
|----------|---------------|------------------------|---------------------|
| 7        | level 6       | 337 × 675              | ~59 km              |
| 8        | level 7       | 169 × 337              | ~118 km             |
| 9        | level 8       | 84 × 169               | ~236 km             |

8 levels gives a manageable coarsest tile (~169 × 337 pixels) while still preserving meaningful spatial detail at the finest level.

### Coarsening method
`method='mean'` is used for all three variables. For day-of-year metrics (SAD, SDD) this is a spatial average, which is visually reasonable. Override with `'min'` or `'max'` if you want conservative estimates.

### Non-spatial dimension
The `water_year` dimension is preserved at every pyramid level — zarr-layer handles it as a non-spatial dimension that the user can slice interactively.

In [7]:
# SAD/SDD valid range is 1–366 (day of water year); max_consec is 0–366
layer_hints = {
    'SAD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='purples'),
    'SDD_DOWY': ZarrLayerVarConfig(clim=[1, 366],  colormap='reds'),
    'max_consec_snow_days': ZarrLayerVarConfig(clim=[0, 366], colormap='blues'), 
}

In [9]:
N_LEVELS = 6

pyramid = create_pyramid(
    ds_crs,
    levels=N_LEVELS,
    x_dim='x',
    y_dim='y',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # web friendly 500 KB chunks
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

Pyramid(datatree=<xarray.DataTree 'root'>
Group: /
│   Attributes:
│       zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
│       multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
│       proj:code:           PROJCS["unnamed",GEOGCS["Unknown datum based upon th...
│       spatial:dimensions:  ['y', 'x']
│       spatial:transform:   [463.3127165287733, 0.0, -20015109.354005992, 0.0, -...
│       spatial:bbox:        [-20015109.354005992, -10007554.677040007, 20015109....
│       spatial:shape:       [43200, 86400]
│       zarr-layer:          {'SAD_DOWY': {'clim': [1, 366], 'colormap': 'purples...
├── Group: /0
│       Dimensions:               (water_year: 10, y: 43200, x: 86400)
│       Coordinates:
│         * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
│         * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
│         * x                     (x) float64 691kB -2.001e+

In [11]:
shard_shape = pyramid.encoding['/0']['SAD_DOWY']['shards']  # e.g. (1, 1440, 1448)
ds_rechunked = ds_crs.chunk({'water_year': 1, 'y': shard_shape[1], 'x': shard_shape[2]})
ds_rechunked

<xarray.Dataset> Size: 448GB
Dimensions:               (water_year: 10, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
  * wkt                   int64 8B 0
Data variables:
    SDD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 1440, 1448), meta=np.ndarray>
    SAD_DOWY              (water_year, y, x) float32 149GB dask.array<chunksize=(1, 1440, 1448), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 149GB dask.array<chunksize=(1, 1440, 1448), meta=np.ndarray>
Indexes:
    wkt      CRSIndex (crs=PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom sphero ...)
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

In [12]:
N_LEVELS = 6

pyramid = create_pyramid(
    ds_rechunked,
    levels=N_LEVELS,
    x_dim='x',
    y_dim='y',
    method='mean',
    target_chunk_bytes=int(0.5 * 1024 * 1024),  # web friendly 500 KB chunks
    chunks_per_shard=4,
    layer_hints=layer_hints,
)

Pyramid(datatree=<xarray.DataTree 'root'>
Group: /
│   Attributes:
│       zarr_conventions:    [{'schema_url': 'https://raw.githubusercontent.com/z...
│       multiscales:         {'layout': [{'asset': '0', 'transform': {'scale': [1...
│       proj:code:           PROJCS["unnamed",GEOGCS["Unknown datum based upon th...
│       spatial:dimensions:  ['y', 'x']
│       spatial:transform:   [463.3127165287733, 0.0, -20015109.354005992, 0.0, -...
│       spatial:bbox:        [-20015109.354005992, -10007554.677040007, 20015109....
│       spatial:shape:       [43200, 86400]
│       zarr-layer:          {'SAD_DOWY': {'clim': [1, 366], 'colormap': 'purples...
├── Group: /0
│       Dimensions:               (water_year: 10, y: 43200, x: 86400)
│       Coordinates:
│         * water_year            (water_year) int64 80B 2015 2016 2017 ... 2023 2024
│         * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
│         * x                     (x) float64 691kB -2.001e+

In [14]:
# maybe here we can update the encoding? 
int16_fill_value = np.iinfo(np.int16).min  # -32768
INT16_ENCODING = {
    'dtype': 'int16',
    '_FillValue': int16_fill_value,
    'write_empty_chunks': False,
}

# Merge with topozarr's chunk/shard encoding (which only has 'chunks'/'shards')
for level_path, level_enc in pyramid.encoding.items():
    for var_name in level_enc:
        level_enc[var_name].update(INT16_ENCODING)

In [15]:
pyramid.encoding

{'/0': {'SDD_DOWY': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'SAD_DOWY': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'max_consec_snow_days': {'chunks': (1, 360, 362),
   'shards': (1, 1440, 1448),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False}},
 '/1': {'SDD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'SAD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False},
  'max_consec_snow_days': {'chunks': (1, 360, 360),
   'shards': (1, 1440, 1440),
   'dtype': 'int16',
   '_FillValue': -32768,
   'write_empty_chunks': False}},
 '/2': {'SDD_DOWY': {'chunks': (1, 360, 360),
   'shards': (1, 1440,

## 4. Write Pyramid to Zarr

- **Azure**: write to the `uwcryo` storage account under a new prefix so zarr-layer can reach it from the browser

The DataTree hierarchy maps directly to Zarr group hierarchy. In Zarr v3 user attributes live in `zarr.json` (not `.zattrs`):
```
modis_snow_phenology_multiscale.zarr/
├── zarr.json        ← multiscales + layer-hints metadata (Zarr v3)
├── 0/               ← finest level (86 400 × 43 200, native resolution)
├── 1/
├── ...
└── 7/               ← coarsest level (~675 × 337)
```

### Write to Azure as Plain Zarr v3

Write via `zarr.storage.ObjectStore` backed by `obstore.store.AzureStore` (Rust-based) — no Icechunk. The pyramid is a derived, read-only product, so versioning adds no value. Using a plain store also eliminates the Icechunk manifest lookup on every chunk fetch, halving HTTP round-trips for the web map.

`obstore` is used instead of `adlfs` because its Rust `object_store` backend correctly handles Azure suffix byte-range requests (needed for reading zarr v3 shard indices). `adlfs` is kept only for the cleanup step below.

For zarr-layer to reach the store from a browser the container must allow **anonymous blob reads** or use a SAS URL. Set the container access level to *Blob* in the Azure portal (or via `az storage container set-permission`).

In [16]:
MULTISCALE_ROOT_PATH = f"{config.AZURE_CONTAINER}/{config.MULTISCALE_PREFIX}"

# Primary store: obstore-backed ObjectStore for all zarr read/write operations.
# prefix is the within-container path (no container name).
store = ObjectStore(AzureStore(
    container_name=config.AZURE_CONTAINER,
    prefix=config.MULTISCALE_PREFIX,
    account_name=config.AZURE_STORAGE_ACCOUNT,
    sas_key=config.AZURE_STORAGE_SAS_TOKEN,
))

# adlfs fs kept for cleanup (recursive rm) — adlfs uses MULTISCALE_ROOT_PATH which includes container name.
fs = adlfs.AzureBlobFileSystem(
    account_name=config.AZURE_STORAGE_ACCOUNT,
    sas_token=config.AZURE_STORAGE_SAS_TOKEN,
    skip_instance_cache=True,
    use_listings_cache=False,
)

In [17]:
# clean up any existing store at the target location before creating new one. have to use await because of asychronous=True
remove_existing_store = True  # set to True to delete existing store and start fresh (warning: this will delete all existing data in the store!)
if remove_existing_store == True:
    if fs.exists(MULTISCALE_ROOT_PATH):
        fs.rm(MULTISCALE_ROOT_PATH, recursive=True)
        print(f"Deleted existing store at {MULTISCALE_ROOT_PATH}")
    else:
        print("No existing store found, nothing to delete")

Deleted existing store at snowmelt/modis_snow_phenology/modis_snow_phenology_multiscale_v1


In [18]:
# Canonical write: DataTree.to_zarr() writes all levels in one shot.
# May OOM on machines with limited RAM (~278 GB uncompressed across 8 levels);
# if so, use the level-by-level read-back approach in the cell below instead.
t0 = time.perf_counter()
pyramid.dt.to_zarr(store, mode='w', encoding=pyramid.encoding, zarr_format=3, consolidated=False)
print(f"Done in {time.perf_counter() - t0:.0f}s")

/home/jovyan/repos/MODIS_snow_phenology/.pixi/envs/default/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 152.40 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Done in 4292s


### Set Cache-Control headers on all blobs

Set `Cache-Control: public, max-age=31536000` on every blob so browsers and CDNs cache chunks aggressively.

In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from azure.storage.blob import BlobServiceClient, ContentSettings

blob_service = BlobServiceClient(
    account_url=f'https://{config.AZURE_STORAGE_ACCOUNT}.blob.core.windows.net',
    credential=config.AZURE_STORAGE_SAS_TOKEN,
)
container_client = blob_service.get_container_client(config.AZURE_CONTAINER)

def set_cache_control(blob):
    container_client.get_blob_client(blob.name).set_http_headers(
        ContentSettings(cache_control='public, max-age=31536000')
    )

prefix = config.MULTISCALE_PREFIX + '/'
blobs = list(container_client.list_blobs(name_starts_with=prefix))

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(set_cache_control, b): b for b in blobs}
    for i, f in enumerate(as_completed(futures), 1):
        f.result()  # raise if any failed
        print(f'{i}/{len(blobs)} done', end='\r')
print(f'\nSet Cache-Control on {len(blobs)} blobs in {time.perf_counter() - t0:.0f}s')

72489/72489 done
Set Cache-Control on 72489 blobs


## Code graveyard / testing

In [ ]:
# # test cell 1: Inspect the dask graph size and structure per level
# # This shows how many dask tasks each level has and confirms the chaining (level 7 should have ~7× the tasks of level 1).
# for i in range(6):
#     ds = pyramid.dt[f"/{i}"].ds
#     var = ds['SAD_DOWY']
#     print(f"Level {i}: shape={var.shape}, chunks={var.chunks}, "
#           f"n_tasks={len(var.data.__dask_graph__())}, "
#           f"nbytes_uncompressed={ds.nbytes / 1e9:.1f} GB")

In [ ]:
# # test cell 2: Visualize the task graph for a single variable
# # Render the graph for a small slice so it's legible
# # This shows whether levels share tasks (indicating chaining) or are independent.
# pyramid.dt['/3'].ds['SAD_DOWY'][0, :100, :100].data.visualize(filename='graph_level3.png')

In [ ]:
# # test cell 3: Profile memory and CPU during a write with dask diagnostics
# # Opens an HTML report with task timeline, memory over time, and cache reuse. This is the most informative for bottleneck diagnosis.

# from dask.diagnostics import Profiler, ResourceProfiler, CacheProfiler, visualize

# with Profiler() as prof, ResourceProfiler(dt=0.5) as rprof, CacheProfiler() as cprof:
#     pyramid.dt['/1'].ds.to_zarr(
#         store=fs.get_mapper(f"{MULTISCALE_ROOT_PATH}/test_1"),
#         mode="w", encoding=pyramid.encoding["/1"],
#         zarr_format=3, consolidated=False
#     )

# visualize([prof, rprof, cprof], filename='profile_level1.html', show=False)

In [ ]:
# # test cell 4: Use dask distributed dashboard (richest view)
# # Then run your write. The dashboard shows exactly which tasks are slow, where memory spikes, and which workers are idle.
# from dask.distributed import Client, LocalCluster
# cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='16GB')
# client = Client(cluster)
# print(client.dashboard_link)  # open in browser → task stream, memory, workers

In [ ]:
# # test cell 5: Quick estimate of recomputation overhead
# # If level i has i × (level 0 tasks), you're seeing full chain recomputation. If all levels have similar task counts, 
# # topozarr may be smarter than expected. The ResourceProfiler approach (#3) will give you the most direct answer about 
# # whether you're I/O-bound (network to Azure) or compute-bound (coarsening), which is the key question for choosing between
# # options A/B/C.
# # Count tasks per level to see if chaining multiplies work
# for i in range(8):
#     n = len(pyramid.dt[f'/{i}'].ds['SAD_DOWY'].data.__dask_graph__())
#     print(f"Level {i}: {n} tasks")

In [ ]:
# diagnostics for dask graph
# import dask

# # Wrap all arrays in a single delayed to get the merged unique task graph
# result = dask.delayed(lambda *x: None)(*arrays)
# combined = result.__dask_graph__()
# print(f"Combined graph: {len(combined)} tasks")

# import pickle, sys
# graph = pyramid.dt['/0'].ds['SAD_DOWY'].data.__dask_graph__()
# print(f"Level 0 SAD_DOWY graph: {len(graph)} tasks, ~{len(pickle.dumps(dict(graph)))/1e6:.1f} MB serialized")

# great to show number of tasks on pyramid before and after rechunking step
# for i in range(N_LEVELS):
#     ds = pyramid.dt[f'/{i}'].ds
#     for v in ds.data_vars:
#         n = len(ds[v].data.__dask_graph__())
#         print(f"Level {i}, {v}: {n} tasks")


# test out dask graph stuff on smaller region
# # Slice a small region from the already-prepared ds_crs
# import modis_snow_phenology
# status_gdf = modis_snow_phenology.config.get_processing_status_gdf(repo, config.TILE_LIST_PATH,years=config.years)
# v, h = 4, 9
# tile_bbox = status_gdf[(status_gdf["v"] == v) & (status_gdf["h"] == h)].geometry.total_bounds
# bbox = [tile_bbox[0], tile_bbox[1], tile_bbox[2], tile_bbox[3]]
# ds_mini = ds_crs.rio.write_crs(ds.rio.crs).rio.clip_box(*bbox, crs=status_gdf.crs)

# pyramid_mini = create_pyramid(
#     ds_mini,
#     levels=8,
#     x_dim='x',
#     y_dim='y',
#     method='mean',
#     target_chunk_bytes=int(0.5 * 1024 * 1024),
#     chunks_per_shard=4,
#     layer_hints=layer_hints,
# )

# int16_fill_value = np.iinfo(np.int16).min  # -32768
# INT16_ENCODING = {
#     'dtype': 'int16',
#     '_FillValue': int16_fill_value,
#     'write_empty_chunks': False,
# }

# # Merge same int16 encoding
# for level_path, level_enc in pyramid_mini.encoding.items():
#     for var_name in level_enc:
#         level_enc[var_name].update(INT16_ENCODING)

# test_store = ObjectStore(AzureStore(
#     container_name=config.AZURE_CONTAINER,
#     prefix=f"{config.MULTISCALE_PREFIX}_test",
#     account_name=config.AZURE_STORAGE_ACCOUNT,
#     sas_key=config.AZURE_STORAGE_SAS_TOKEN,
# ))

# import time, psutil

# # Snapshot before
# mem_before = psutil.virtual_memory().used / 1e9

# t0 = time.perf_counter()
# pyramid_mini.dt.to_zarr(
#     test_store, mode='w', encoding=pyramid_mini.encoding,
#     zarr_format=3, consolidated=False
# )
# elapsed = time.perf_counter() - t0

# # Snapshot after
# mem_after = psutil.virtual_memory().used / 1e9

# print(20*'-')

# print(f"CPUs: {os.cpu_count()}")
# print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.0f} GB")
# print(f"Available RAM now: {psutil.virtual_memory().available / 1e9:.0f} GB")

# ncores = client.nthreads()   # {worker_address: n_threads}
# print(f"Workers:       {len(ncores)}")
# print(f"Total threads: {sum(ncores.values())}")
# print(f"Threads/worker: {set(ncores.values())}") 

# print(f"Zarr config: async.concurrency: {zarr.config.get('async.concurrency')}")
# print(f"Zarr config threading.max_workers: {zarr.config.get('threading.max_workers')}")
# print(20*'-')

# uncompressed_gb = sum(pyramid_mini.dt[f'/{i}'].ds.nbytes for i in range(8)) / 1e9
# print(f"Time:          {elapsed:.1f}s")
# print(f"Uncompressed:  {uncompressed_gb:.2f} GB")
# print(f"Throughput:    {uncompressed_gb / elapsed * 1000:.0f} MB/s (uncompressed)")
# print(f"Memory delta:  {mem_after - mem_before:.1f} GB")

# spill_info = client.run(lambda dask_worker: {
#     'memory_mb': dask_worker.monitor.get_process_memory() / 1e6,
#     'tasks_done': dask_worker.state.executed_count,
# })
# for addr, info in spill_info.items():
#     print(f"  worker {addr[-5:]}: {info['memory_mb']:.0f} MB,  {info['tasks_done']} tasks")


In [ ]:
# # ── Option C: level-by-level read-back (uncomment if canonical approach OOMs) ──────────
# #
# # Writes level 0 from the lazy pyramid, then reads each written level back from Azure
# # to coarsen and write the next. Avoids holding the full pyramid in memory (~278 GB).
# # Uses obstore so suffix byte-range requests (needed for shard-index reads) work correctly.
# #
# def make_level_store(i):
#     return ObjectStore(AzureStore(
#         container_name=config.AZURE_CONTAINER,
#         prefix=f"{MULTISCALE_PREFIX}/{i}",
#         account_name=config.AZURE_STORAGE_ACCOUNT,
#         sas_key=config.AZURE_STORAGE_SAS_TOKEN,
#     ))

# def write_pyramid_readback(pyramid, x_dim="x", y_dim="y", zarr_format=3):
#     n_levels = len(pyramid.encoding)

#     root = zarr.open_group(store, mode="w", zarr_format=zarr_format)
#     root.attrs.update(pyramid.dt.attrs)

#     for i in range(n_levels):
#         t0 = time.perf_counter()
#         if i == 0:
#             ds = pyramid.dt["/0"].ds
#         else:
#             prev_ds = xr.open_zarr(make_level_store(i - 1), consolidated=False)
#             ds = prev_ds.coarsen(dim={x_dim: 2, y_dim: 2}, boundary="trim").mean()

#         ny, nx = ds.sizes.get(y_dim, "?"), ds.sizes.get(x_dim, "?")
#         print(f"[{i+1}/{n_levels}] Writing level {i}  ({ny} x {nx})", flush=True)
#         ds.to_zarr(
#             store=make_level_store(i),
#             mode="w",
#             encoding=pyramid.encoding[f"/{i}"],
#             zarr_format=zarr_format,
#             consolidated=False,
#             align_chunks=True,
#         )
#         print(f"[{i+1}/{n_levels}] Level {i} done  ({time.perf_counter()-t0:.1f}s)", flush=True)

# with ProgressBar(dt=30):
#     write_pyramid_readback(pyramid, x_dim='x', y_dim='y')